# CodeAlpha Data Analytics Internship
## Task 2 — Exploratory Data Analysis (EDA)

**Dataset:** `remote_data_science_jobs.csv` — collected via web scraping (Task 1)

This notebook covers understanding the data's structure, cleaning it, spotting issues, and asking/answering meaningful questions about it.


## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

df = pd.read_csv('data/remote_data_science_jobs.csv')
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'remote_data_science_jobs.csv'

## 2. Understand the Structure
What columns do we have, what type of data is in each, and how much of it is missing?

In [ ]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   scrape_category  250 non-null    str  
 1   job_title        250 non-null    str  
 2   company          231 non-null    str  
 3   location         147 non-null    str  
 4   category         249 non-null    str  
 5   job_level        250 non-null    str  
 6   salary           100 non-null    str  
 7   published_date   250 non-null    str  
 8   job_url          250 non-null    str  
dtypes: str(9)
memory usage: 17.7 KB


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary[missing_summary['missing_count'] > 0]


,missing_count,missing_pct
salary,150,60.0
location,103,41.2
company,19,7.6
category,1,0.4


**Observations so far:**
- `salary` is missing for a large share of postings — many companies simply don't disclose pay. We should treat "no salary listed" as a signal, not just drop those rows.
- `location` and `company` also have some gaps.
- `scrape_category` looks like it might be a constant (single value) leftover from the scraper — worth checking.


In [ ]:
for col in ['scrape_category', 'category', 'job_level']:
    print(f"--- {col} ({df[col].nunique()} unique values) ---")
    print(df[col].value_counts())
    print()


--- scrape_category (1 unique values) ---
scrape_category
Data    250
Name: count, dtype: int64

--- category (11 unique values) ---
category
📊 Data                    213
💻 Software Development     12
🚀 Product                   9
🔒 Cybersecurity             4
🔧 DevOps                    3
🔍 QA                        3
📋 Project Management        1
🎨 Design                    1
🏢 Business                  1
⚖️ Finance & Legal          1
📝 Writing                   1
Name: count, dtype: int64

--- job_level (8 unique values) ---
job_level
🟣 Senior         96
🔵 Mid-level      91
🟠 Manager        22
🟡 Principal      15
🔴 Director       15
🟤 Staff           6
🟢 Entry Level     3
⚫ Executive       2
Name: count, dtype: int64



`scrape_category` only ever equals `"Data"` — it's a constant from the scraping run, not a useful analytical column, so we'll drop it. `category` and `job_level` are the more informative categorical fields, so those become our main grouping variables.

In [ ]:
df = df.drop(columns=['scrape_category'])


## 3. Clean & Engineer Features
Two columns need work before they're analysis-ready:
- `salary` is a free-text range like `"$93k-$118k"` or a single value like `"$42k"` — we'll parse it into numeric `salary_min`, `salary_max`, and `salary_avg` (in USD thousands).
- `published_date` is an ISO timestamp string — we'll convert it to a real datetime and pull out the month.


In [ ]:
import re

def parse_salary(val):
    """Parse strings like '$93k-$118k' or '$42k' into (min, max) in $k."""
    if pd.isna(val):
        return (np.nan, np.nan)
    nums = re.findall(r'\$(\d+)k', str(val))
    nums = [int(n) for n in nums]
    if len(nums) == 2:
        return (nums[0], nums[1])
    elif len(nums) == 1:
        return (nums[0], nums[0])
    return (np.nan, np.nan)

df[['salary_min', 'salary_max']] = df['salary'].apply(lambda v: pd.Series(parse_salary(v)))
df['salary_avg'] = df[['salary_min', 'salary_max']].mean(axis=1)
df['has_salary'] = df['salary'].notna()

df['published_date'] = pd.to_datetime(df['published_date'], errors='coerce')
df['published_month'] = df['published_date'].dt.to_period('M').astype(str)

# Clean flag emoji off location/job_level/category for cleaner labels in charts later
df['location_clean'] = df['location'].str.replace(r'^[^\w]+', '', regex=True).str.strip()
df['job_level_clean'] = df['job_level'].str.replace(r'^[^\w]+', '', regex=True).str.strip()
df['category_clean'] = df['category'].str.replace(r'^[^\w]+', '', regex=True).str.strip()

df[['salary', 'salary_min', 'salary_max', 'salary_avg', 'has_salary', 'published_date', 'published_month']].head(10)


/tmp/ipykernel_87/3320824841.py:20: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['published_month'] = df['published_date'].dt.to_period('M').astype(str)


,salary,salary_min,salary_max,salary_avg,has_salary,published_date,published_month
0,$93k-$118k,93.0,118.0,105.5,True,2026-08-15 22:48:33+00:00,2026-08
1,NaN,NaN,NaN,NaN,False,2026-08-12 11:02:38+00:00,2026-08
2,$170k-$185k,170.0,185.0,177.5,True,2026-08-12 07:28:01+00:00,2026-08
3,NaN,NaN,NaN,NaN,False,2026-08-01 06:50:59+00:00,2026-08
4,NaN,NaN,NaN,NaN,False,2026-07-25 03:02:37+00:00,2026-07
5,NaN,NaN,NaN,NaN,False,2026-07-23 17:16:17+00:00,2026-07
6,NaN,NaN,NaN,NaN,False,2026-07-22 23:00:49+00:00,2026-07
7,NaN,NaN,NaN,NaN,False,2026-07-08 22:29:14+00:00,2026-07
8,$75k-$95k,75.0,95.0,85.0,True,2026-06-15 20:55:28+00:00,2026-06
9,NaN,NaN,NaN,NaN,False,2026-06-12 13:49:21+00:00,2026-06


## 4. Descriptive Statistics

In [ ]:
df[['salary_min', 'salary_max', 'salary_avg']].describe()


,salary_min,salary_max,salary_avg
count,100.000000,100.000000,100.000000
mean,138.620000,175.270000,156.945000
std,50.383335,63.987034,56.487502
min,20.000000,20.000000,20.000000
25%,99.000000,124.750000,112.875000
50%,139.500000,180.000000,160.000000
75%,175.000000,215.000000,194.125000
max,265.000000,310.000000,279.000000


## 5. Asking Meaningful Questions

Now that the data is clean, let's explore some real questions:

1. Which job categories and seniority levels dominate the remote data-science market right now?
2. Where in the world are these remote roles based?
3. How does pay vary by seniority level?
4. Is there a relationship between seniority and how likely a company is to disclose salary?
5. Has posting volume changed over time?


In [ ]:
print("Q1: Job postings by category")
print(df['category_clean'].value_counts())
print()
print("Job postings by seniority level")
print(df['job_level_clean'].value_counts())


Q1: Job postings by category
category_clean
Data                    213
Software Development     12
Product                   9
Cybersecurity             4
DevOps                    3
QA                        3
Project Management        1
Design                    1
Business                  1
Finance & Legal           1
Writing                   1
Name: count, dtype: int64

Job postings by seniority level
job_level_clean
Senior         96
Mid-level      91
Manager        22
Principal      15
Director       15
Staff           6
Entry Level     3
Executive       2
Name: count, dtype: int64


In [ ]:
print("Q2: Top 10 countries for remote data-science roles")
print(df['location_clean'].value_counts().head(10))
print(f"\n{df['location'].isna().sum()} postings ({df['location'].isna().mean()*100:.1f}%) don't list a location at all.")


Q2: Top 10 countries for remote data-science roles
location_clean
United States     70
Canada            18
United Kingdom    10
Brazil             8
India              7
Poland             5
Spain              4
Slovakia           3
Mexico             3
Hungary            2
Name: count, dtype: int64

103 postings (41.2%) don't list a location at all.


In [ ]:
print("Q3: Average salary (in $k) by seniority level, where disclosed")
salary_by_level = (
    df.dropna(subset=['salary_avg'])
      .groupby('job_level_clean')['salary_avg']
      .agg(['mean', 'median', 'count'])
      .sort_values('mean')
)
salary_by_level.round(1)


Q3: Average salary (in $k) by seniority level, where disclosed


,mean,median,count
job_level_clean,,,
Mid-level,130.8,125.0,32
Senior,153.4,151.5,36
Principal,165.8,182.0,7
Manager,166.8,182.2,12
Director,211.2,205.0,9
Staff,220.2,245.0,3
Executive,263.0,263.0,1


In [ ]:
print("Q4: Salary disclosure rate by seniority level")
disclosure_by_level = df.groupby('job_level_clean')['has_salary'].mean().sort_values(ascending=False) * 100
disclosure_by_level.round(1)


Q4: Salary disclosure rate by seniority level


job_level_clean
Director       60.0
Manager        54.5
Executive      50.0
Staff          50.0
Principal      46.7
Senior         37.5
Mid-level      35.2
Entry Level     0.0
Name: has_salary, dtype: float64

In [ ]:
print("Q5: Postings per month")
df['published_month'].value_counts().sort_index()


Q5: Postings per month


published_month
2026-03      1
2026-04      3
2026-05      2
2026-06     17
2026-07     10
2026-08    217
Name: count, dtype: int64

## 6. Data Quality Issues Found

- **Salary transparency gap:** ~60% of postings don't list a salary at all, so any salary-based conclusion only reflects the subset of employers willing to disclose pay — likely a biased sample, not the whole market.
- **Missing locations:** roughly 40% of postings have no location, which limits how confidently we can say *where* remote roles are concentrated.
- **Emoji-prefixed categorical values:** `location`, `category`, and `job_level` all had flag/emoji prefixes baked into the text (scraper artifact) — cleaned into `*_clean` columns above so charts and groupings aren't split by cosmetic differences.
- **Redundant constant column:** `scrape_category` carried no information (always `"Data"`) and was dropped.
- **Small sample in some slices:** levels like `Entry Level` (n=3) and `Executive` (n=2) have too few postings for statistically reliable averages — flagged so we don't over-interpret them in the visuals.


## 8. Save the Cleaned Dataset
Export the cleaned, feature-engineered dataframe so Task 3 (and any future analysis) can load it directly instead of re-cleaning the raw file.

In [ ]:
df.to_csv('cleaned_remote_data_science_jobs.csv', index=False)
print(f"Saved cleaned_remote_data_science_jobs.csv -> {df.shape[0]} rows x {df.shape[1]} columns")
df.head()


Saved cleaned_remote_data_science_jobs.csv -> 250 rows x 16 columns


,job_title,company,location,category,job_level,salary,published_date,job_url,salary_min,salary_max,salary_avg,has_salary,published_month,location_clean,job_level_clean,category_clean
0,"Data Engineer, Event Data",Movable Ink,🇨🇦 Canada,📊 Data,🔵 Mid-level,$93k-$118k,2026-08-15 22:48:33+00:00,https://remotefirstjobs.com/companies/movable-...,93.0,118.0,105.5,True,2026-08,Canada,Mid-level,Data
1,Data Engineer (Domain Data),OLX,🇵🇹 Portugal,📊 Data,🔵 Mid-level,NaN,2026-08-12 11:02:38+00:00,https://remotefirstjobs.com/companies/olx/jobs...,NaN,NaN,NaN,False,2026-08,Portugal,Mid-level,Data
2,Data Lead - Central Data Team,YipitData,🇺🇸 United States,📊 Data,🟠 Manager,$170k-$185k,2026-08-12 07:28:01+00:00,https://remotefirstjobs.com/companies/yipitdat...,170.0,185.0,177.5,True,2026-08,United States,Manager,Data
3,Data Architect and Data Modeller,Riverflex,NaN,📊 Data,🟡 Principal,NaN,2026-08-01 06:50:59+00:00,https://remotefirstjobs.com/companies/riverfle...,NaN,NaN,NaN,False,2026-08,NaN,Principal,Data
4,Senior Data Engineer (AI-Native) — Data Layer,Proton.ai,NaN,📊 Data,🟣 Senior,NaN,2026-07-25 03:02:37+00:00,https://remotefirstjobs.com/companies/protonai...,NaN,NaN,NaN,False,2026-07,NaN,Senior,Data


## 7. EDA Summary

- The dataset is skewed toward **Data**-category roles at **Mid-level** and **Senior** seniority, based mostly in the **United States**.
- Salary is under-reported, so pay comparisons should be read as directional, not definitive.
- With cleaning done, the data is ready for the visualizations in Task 3 below.
